# RouterGuard — Step 4: redundancy-aware pruning and the 2x2 ablation

**Gate for Step 4:** does the redundancy ranking beat frequency pruning by
`min_gain_over_frequency_pts` perplexity points? Either answer is reportable — if it does not
beat frequency, the controlled negative result *is* the deliverable, as stated in the protocol.

Design being tested (2x2, from `protocol.yaml:design.ablation`):

|  | frequency ranking | redundancy ranking |
|---|---|---|
| **unbalanced router** | baseline | main claim |
| **balanced router** | control | main claim + balance |

The comparison that matters for the paper's gap claim is *redundancy ranked under a rebalanced
router*, which prior work (arXiv:2402.14800) never does.

Cost control: similarity needs every expert to run on every token, which is `n_experts / top_k`
(~8x) the routed cost. `budget.max_tokens_per_layer` and the timing probe below bound it, and
matrices are cached per layer so a killed session resumes.

In [ ]:
# 00 — install (skip if a previous notebook already installed and you restarted)
INSTALL = False
if INSTALL:
    import subprocess, sys
    pkgs = [
        "transformers==4.48.3",
        "huggingface-hub>=0.24,<1.0",
        "datasets>=2.19,<5",
        "accelerate>=0.34",
        "bitsandbytes>=0.43.3",
        "pyyaml", "matplotlib", "sentencepiece",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    print("installed; RESTART the runtime before continuing")

In [ ]:
# 01 — bootstrap
import os, sys, json, gc, time
from pathlib import Path

ROOT = Path(os.environ.get("ROUTERGUARD_ROOT", "/content/routerguard")).expanduser()
if not (ROOT / "routerguard" / "env.py").exists():
    for candidate in (Path.home() / "routerguard", Path.cwd(), Path.cwd().parent):
        if (candidate / "routerguard" / "env.py").exists():
            ROOT = candidate
            break
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

# Similarity under the *balanced* router doubles the similarity cost. Turn it off only
# if you accept that the 2x2's balanced cells then rank with unbalanced similarity.
SIM_UNDER_BALANCE = True

In [ ]:
# 02 — environment, run, protocol
import torch
from routerguard import env, models, data, stats, moe, balance, prune, similarity, eval as rgeval, run as rgrun

report = env.assert_env(strict=True)
print(f"transformers {report['versions']['transformers']} | gpu {report['gpu']['name']} ({report['gpu']['total_gb']} GB)")

PROTOCOL = rgrun.load_protocol(ROOT / "configs" / "protocol.yaml")
RUN = rgrun.new_run("step4_ablation", protocol=PROTOCOL)
rgrun.set_seed(RUN.seed)
print(RUN)

In [ ]:
# 03 — model, splits, and load counts on calibration data
model, tokenizer, info = models.load_olmoe(precision="auto")
print(info)
RUN.save_json("load_info", info.to_dict())

splits = data.build_splits(tokenizer, PROTOCOL, cache_dir=ROOT / "results" / "splits")
calib_ids = splits["calib"].input_ids.to("cuda")
eval_ids = splits["eval"].input_ids.to("cuda")
print(splits["calib"].describe())
print(splits["eval"].describe())

baseline_ppl = rgeval.wikitext_perplexity(model, eval_ids, batch_size=1, device="cuda")
print(f"baseline ppl = {baseline_ppl['perplexity']:.4f}")
RUN.save_json("baseline_ppl", baseline_ppl)

unbalanced_counts = balance.measure_model(model, calib_ids, None, batch_size=4, device="cuda")
print(f"unbalanced calib gini = {unbalanced_counts['gini_mean']:.4f}")

In [ ]:
# 04 — fit the calibration biases once, freeze, and measure load under them
bias_result = balance.evaluate_balance(
    model, calib_ids, eval_ids, PROTOCOL, batch_size=4, device="cuda", progress=True
)
for w in bias_result.warnings():
    print(f"WARNING: {w}")
print(f"gini calib {bias_result.calib_before['gini_mean']:.4f} -> {bias_result.calib_after['gini_mean']:.4f}"
      f" | eval {bias_result.eval_before['gini_mean']:.4f} -> {bias_result.eval_after['gini_mean']:.4f}")
print(f"transfer ratio {bias_result.transfer_ratio:.3f}")

biases = bias_result.biases
balanced_counts = bias_result.calib_after
RUN.save_json("balance", bias_result.summary())
bias_result.save(RUN.dir / "balance_biases.pt")

In [ ]:
# 05 — timing probe before spending the budget: 2 layers, 2 batches, then project.
probe_batches = 2
probe_layer_count = 2
t0 = time.time()
with similarity.SimilarityCollector(model, max_tokens_per_layer=PROTOCOL["budget"]["max_tokens_per_layer"]) as probe:
    for start in range(0, min(probe_batches * 4, calib_ids.shape[0]), 4):
        with torch.no_grad():
            model(input_ids=calib_ids[start : start + 4], use_cache=False)
    probe_matrices = probe.matrices()
probe_s = time.time() - t0
batches_total = (calib_ids.shape[0] + 3) // 4
projected_s = probe_s / probe_batches * batches_total
print(f"probe: {probe_s:.1f}s for {probe_batches} batches of 4 seqs -> projected {projected_s/60:.1f} min per router state")
if SIM_UNDER_BALANCE:
    print(f"projected total for both router states: {2 * projected_s / 60:.1f} min")

cap = float(PROTOCOL["budget"]["abort_if_projected_over_seconds"])
print(f"cap = {cap:.0f}s per state")
if projected_s > cap:
    raise SystemExit(
        f"ABORT: projected {projected_s:.0f}s > cap {cap:.0f}s. Lower budget.max_tokens_per_layer "
        "or run this on a larger GPU; do not silently proceed."
    )
RUN.save_json("similarity_timing_probe", {"probe_s": probe_s, "projected_s": projected_s, "cap_s": cap})

In [ ]:
# 06 — similarity matrices, one forward pass, cached per layer
sim_dir = ROOT / "results" / "sim" / "unbalanced"
sim_report = similarity.build_similarity(
    model, calib_ids, sim_dir,
    max_tokens_per_layer=int(PROTOCOL["budget"]["max_tokens_per_layer"]),
    batch_size=1, device="cuda",
)
sim_unbalanced = sim_report.matrices
print(f"tokens/layer = {sorted(set(sim_report.tokens_per_layer.values()))} | {sim_report.elapsed_s:.1f}s")
RUN.save_json("similarity_unbalanced", sim_report.to_dict())

for layer in sorted(sim_unbalanced)[:3]:
    top = similarity.redundancy_pairs(sim_unbalanced[layer], top=3)
    print(f"  layer {layer} most redundant pairs: " + ", ".join(f"({i},{j}) {s:.3f}" for i, j, s in top))

In [ ]:
# 07 — similarity under the balanced router (only if the router actually moved)
# Similarity depends on layer inputs, which shift when the router is rebalanced, so the
# biases must be ACTIVE while collecting. But if the bias barely changes routing, the
# matrices are identical and the pass is wasted GPU time — so this is a measured decision.
SIM_RECOMPUTE_KL = 1e-2   # nats; below this, recomputing similarity is not worth ~8 min
sim_balanced = sim_unbalanced
routing_kl = {
    int(layer): stats.kl(unbalanced_counts["per_layer_counts"][layer], balanced_counts["per_layer_counts"][layer])
    for layer in unbalanced_counts["per_layer_counts"]
}
mean_kl = sum(routing_kl.values()) / len(routing_kl)
print(f"router change under bias: mean KL {mean_kl:.5f} nats, max {max(routing_kl.values()):.5f}")

if SIM_UNDER_BALANCE and mean_kl >= SIM_RECOMPUTE_KL:
    with torch.no_grad(), balance.apply_bias(model, biases, device="cuda"):
        sim_balanced = similarity.build_similarity(
            model, calib_ids, ROOT / "results" / "sim" / "balanced",
            max_tokens_per_layer=int(PROTOCOL["budget"]["max_tokens_per_layer"]),
            batch_size=1, device="cuda",
        ).matrices
    drift = {
        int(layer): float(abs(sim_balanced[layer] - sim_unbalanced[layer]).mean())
        for layer in sorted(sim_unbalanced)
    }
    print("mean |cos(balanced) - cos(unbalanced)| per layer:",
          {k: round(v, 4) for k, v in list(drift.items())[:5]})
    print("  (layer 0 is router-invariant by construction: its inputs are the embeddings)")
    RUN.save_json("similarity_balanced", {"mean_kl": mean_kl, "drift": drift, "recomputed": True})
else:
    print(f"SKIPPING similarity recompute: router moved < {SIM_RECOMPUTE_KL} nats; "
          "balanced cells reuse the unbalanced matrices (recorded)")
    RUN.save_json("similarity_balanced", {"mean_kl": mean_kl, "recomputed": False})

In [ ]:
# 08 — the 2x2: {unbalanced, balanced} x {frequency, redundancy} at 25% and 50%
utils = [float(u) for u in [PROTOCOL["design"].get("utilization_weight", 0.25)]]
ratios = [float(r) for r in PROTOCOL["pruning"]["ratios"]]

cells = []
for router_state in ("unbalanced", "balanced"):
    bias = None if router_state == "unbalanced" else biases
    counts = unbalanced_counts if router_state == "unbalanced" else balanced_counts
    sim = sim_unbalanced if router_state == "unbalanced" else sim_balanced
    for ranking in ("frequency", "redundancy"):
        for ratio in ratios:
            keeps = prune.keeps_for(
                model, ranking, ratio,
                per_layer_counts=counts["per_layer_counts"],
                sim=sim, utilization_weight=utils[0],
                max_steps=int(PROTOCOL["budget"]["max_greedy_steps"]),
            )
            with torch.no_grad(), moe.MaskSim(model, keeps, bias=bias):
                ppl = rgeval.wikitext_perplexity(model, eval_ids, batch_size=1, device="cuda")
            point = {
                "router": router_state, "ranking": ranking, "ratio": ratio,
                "perplexity": ppl["perplexity"], "kept_per_layer": len(keeps[sorted(keeps)[0]]),
                "delta_vs_baseline": ppl["perplexity"] - baseline_ppl["perplexity"],
            }
            cells.append(point)
            print(f"{router_state:10} {ranking:10} ratio={ratio:.2f} kept={point['kept_per_layer']:3d} "
                  f"ppl={ppl['perplexity']:.4f}  ({point['delta_vs_baseline']:+.4f} vs baseline)")

RUN.save_json("ablation_2x2", {"baseline_ppl": baseline_ppl["perplexity"], "cells": cells})

In [ ]:
# 09 — Step 4 gate: redundancy must beat frequency by min_gain_over_frequency_pts
rows = {(c["router"], c["ranking"], c["ratio"]): c["perplexity"] for c in cells}
gains = {}
for router_state in ("unbalanced", "balanced"):
    for ratio in ratios:
        gain = rows[(router_state, "frequency", ratio)] - rows[(router_state, "redundancy", ratio)]
        gains[f"{router_state}@{ratio:.2f}"] = gain
        print(f"{router_state:10} ratio={ratio:.2f}  freq {rows[(router_state,'frequency',ratio)]:.4f}"
              f"  redundancy {rows[(router_state,'redundancy',ratio)]:.4f}  gain {gain:+.4f} ppl points")

best_router = "balanced" if SIM_UNDER_BALANCE else "unbalanced"
key = f"{best_router}@{max(ratios):.2f}"
measured = {"min_gain_over_frequency_pts": gains[key]}
gates = rgrun.summarize_gates(PROTOCOL, measured)
print()
print(rgrun.format_gates(gates))
RUN.save_json("step4_gates", {"gains": gains, "gates": gates, "headline_cell": key})

verdict = (
    "REDUNDANCY WINS - main claim supported"
    if gains[key] >= PROTOCOL["gates"]["min_gain_over_frequency_pts"]
    else "NEGATIVE RESULT - redundancy does not beat frequency; this is the deliverable"
)
print()
print(verdict)

In [ ]:
# 10 — plot the 2x2
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 4))
styles = {("unbalanced", "frequency"): "o:", ("unbalanced", "redundancy"): "o-",
          ("balanced", "frequency"): "s:", ("balanced", "redundancy"): "s-"}
for router_state in ("unbalanced", "balanced"):
    for ranking in ("frequency", "redundancy"):
        xs = [0.0] + [r * 100 for r in ratios]
        ys = [baseline_ppl["perplexity"]] + [rows[(router_state, ranking, r)] for r in ratios]
        ax.plot(xs, ys, styles[(router_state, ranking)], label=f"{router_state} / {ranking}")
ax.set_xlabel("routed experts removed (%)")
ax.set_ylabel("wikitext perplexity")
ax.set_title("2x2 ablation: balance x redundancy")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RUN.path("figures", "step4_ablation_2x2.png"), dpi=130, bbox_inches="tight")
plt.show()
print("saved:", RUN.path("figures", "step4_ablation_2x2.png"))

## Step 4 gate

- [ ] similarity matrices cached for every layer, with token counts under the budget
- [ ] timing probe projected under `abort_if_projected_over_seconds`
- [ ] all 8 ablation cells measured (2 router states x 2 rankings x 2 ratios)
- [ ] `min_gain_over_frequency_pts` evaluated on the *balanced* router cell at 50%
- [ ] verdict recorded, in either direction

If the gain is negative or below 1.0 point, do **not** retune the ranking to rescue it. Write the
negative result: prior work's utilization-only ranking is apparently sufficient at these ratios,
and the "redundancy" contribution is null at 25–50% removal on OLMoE. That is a legitimate finding
and the protocol committed to reporting it.

Step 5 ports the winning configuration to Qwen1.5-MoE (4-bit), with a timing probe first.